# TGB Modelling

## Load the Full Dataset
The full dataset is 1.2Gb as a parquet file, with 700+ columns. Care is needed to ensure memory constraints are respected. 

In [1]:
import pandas as pd
import gc

df_full = pd.read_parquet("../Data/EastAfrica Preprocessed/eastafrica-ready-for-tgb.parquet")

In [2]:
#convert float64s to float32 to save memory:
float_cols = df_full.select_dtypes(include=["float64"]).columns

df_full[float_cols] = df_full[float_cols].astype("float32")


In [3]:
print("Shape: ", df_full.shape)

for col in df_full.columns:
    print (f"{col}: ", df_full[col].dtype)


Shape:  (2314906, 755)
id:  category
name:  category
country:  category
latitude:  float32
longitude:  float32
minBorderDistanceKm:  float32
minCapitalDistanceKm:  float32
periodStart:  datetime64[ns]
month_sin:  float32
month_cos:  float32
is_voting:  bool
TargetConflictLocal_t+1:  boolean
TargetConflictRegional_t+1:  boolean
TargetConflictLocal_t+3:  boolean
TargetConflictRegional_t+3:  boolean
TargetConflictLocal_t+6:  boolean
TargetConflictRegional_t+6:  boolean
TargetConflictLocal_t+12:  boolean
TargetConflictRegional_t+12:  boolean
LocalEventCount_std1:  float32
LocalEventCount_std2:  float32
LocalEventCount_std3:  float32
LocalEventCount_std6:  float32
LocalEventCount_std12:  float32
LocalEventCount_std18:  float32
LocalDistinctActorCount_std1:  float32
LocalDistinctActorCount_std2:  float32
LocalDistinctActorCount_std3:  float32
LocalDistinctActorCount_std6:  float32
LocalDistinctActorCount_std12:  float32
LocalDistinctActorCount_std18:  float32
LocalDistinctEventTypes_std1:  f

In [4]:
#define target columns, we'll iterate over this list later. These need to be exact matches to stop target information leaking into the training (looking at you, AUC = 1) 
target_columns = [
"TargetConflictLocal_t+1",
"TargetConflictRegional_t+1",
"TargetConflictLocal_t+3",
"TargetConflictRegional_t+3",
"TargetConflictLocal_t+6",
"TargetConflictRegional_t+6",
"TargetConflictLocal_t+12",
"TargetConflictRegional_t+12"
]

The below builds a list of columns we will train with, we can know these in advance: most of the columns are lagged, a small number (country, latitude, longitude, country, minBorderDistanceKm, minCapitalDistanceKm, is_voting) are either static for the place, or can be known in advance)

In [5]:
#Drop columns named median: suspect leakage.
df_full = df_full.drop(columns=df_full.filter(regex="Median").columns)
df_full = df_full.drop(columns=df_full.filter(regex="median").columns)

In [6]:

feature_cols = [
    c for c in df_full.columns
    if c not in target_columns
    and c not in ["id", 'name',"periodStart", "periodEnd", "period_midpoint"]
]
print(feature_cols)

['country', 'latitude', 'longitude', 'minBorderDistanceKm', 'minCapitalDistanceKm', 'month_sin', 'month_cos', 'is_voting', 'LocalEventCount_std1', 'LocalEventCount_std2', 'LocalEventCount_std3', 'LocalEventCount_std6', 'LocalEventCount_std12', 'LocalEventCount_std18', 'LocalDistinctActorCount_std1', 'LocalDistinctActorCount_std2', 'LocalDistinctActorCount_std3', 'LocalDistinctActorCount_std6', 'LocalDistinctActorCount_std12', 'LocalDistinctActorCount_std18', 'LocalDistinctEventTypes_std1', 'LocalDistinctEventTypes_std2', 'LocalDistinctEventTypes_std3', 'LocalDistinctEventTypes_std6', 'LocalDistinctEventTypes_std12', 'LocalDistinctEventTypes_std18', 'LocalDistinctEventSubtypes_std1', 'LocalDistinctEventSubtypes_std2', 'LocalDistinctEventSubtypes_std3', 'LocalDistinctEventSubtypes_std6', 'LocalDistinctEventSubtypes_std12', 'LocalDistinctEventSubtypes_std18', 'LocalTotalFatalities_std1', 'LocalTotalFatalities_std2', 'LocalTotalFatalities_std3', 'LocalTotalFatalities_std6', 'LocalTotalFata

## Define Training and testing datasets
Tree based models do not require a full series (of Ids) to be provided, or for full series to be present in the training data: we can split as required. 

### Removing records with NaN
We can drop the first 18 periods, as they will have NaN for the longest window. 

In [7]:
print("Shape before dropping: ", df_full.shape)
df_full = df_full[df_full['LocalEventCount_std18'].notna()].copy()
print("Shape after dropping: ", df_full.shape)

Shape before dropping:  (2314906, 575)
Shape after dropping:  (2298264, 575)


We can also drop the last 12 periods. These will have False or NaN as their value in the t+12 target columns. That's not useful for actually comparing the performance of the models. 

In [8]:
print("Shape before dropping: ", df_full.shape)
all_period_starts = sorted(df_full['periodStart'].unique(), reverse = True)
invalid_target_periods = all_period_starts[:12]
print("Invalid periods: ",invalid_target_periods )
df_full = df_full[~df_full['periodStart'].isin(invalid_target_periods)].copy()
print("Shape after dropping: ", df_full.shape)


Shape before dropping:  (2298264, 575)
Invalid periods:  [Timestamp('2025-07-31 00:00:00'), Timestamp('2025-07-03 00:00:00'), Timestamp('2025-06-05 00:00:00'), Timestamp('2025-05-08 00:00:00'), Timestamp('2025-04-10 00:00:00'), Timestamp('2025-03-13 00:00:00'), Timestamp('2025-02-13 00:00:00'), Timestamp('2025-01-16 00:00:00'), Timestamp('2024-12-19 00:00:00'), Timestamp('2024-11-21 00:00:00'), Timestamp('2024-10-24 00:00:00'), Timestamp('2024-09-26 00:00:00')]
Shape after dropping:  (2098416, 575)


### Train/Test Split


#### Build the Testing Set

In [9]:
all_period_starts = sorted(df_full['periodStart'].unique(), reverse = True)
testing_periods = all_period_starts[:12]
print("Testing periods: ", min(testing_periods) , " - ", max(testing_periods))
df_test = df_full[df_full['periodStart'].isin(testing_periods)].copy()
print("Testing shape:", df_test.shape)

Testing periods:  2023-10-26 00:00:00  -  2024-08-29 00:00:00
Testing shape: (199848, 575)


#### Buid the Training set

In [10]:
df_train = df_full[~df_full["periodStart"].isin(testing_periods)].copy()
print("Training shape: ", df_train.shape)
unique_training_periods = df_train['periodStart'].unique()
print("Training periods: ", min(unique_training_periods), ' - ',  max(unique_training_periods))

Training shape:  (1898568, 575)
Training periods:  2015-01-01 00:00:00  -  2023-09-28 00:00:00


In [11]:
del df_full
gc.collect()

0

## LightGBM Training

### Subsampling strategy for 5-fold CV
Every period contributes to the sample. temporal structure is preserved, fold size is controlled at 200K, positives are kept, neagtives are balances across time. All positives are selected, which might not be a good idea since they are much more common than when doing point predictions. 

In [12]:
for col in feature_cols:
    print (col)

country
latitude
longitude
minBorderDistanceKm
minCapitalDistanceKm
month_sin
month_cos
is_voting
LocalEventCount_std1
LocalEventCount_std2
LocalEventCount_std3
LocalEventCount_std6
LocalEventCount_std12
LocalEventCount_std18
LocalDistinctActorCount_std1
LocalDistinctActorCount_std2
LocalDistinctActorCount_std3
LocalDistinctActorCount_std6
LocalDistinctActorCount_std12
LocalDistinctActorCount_std18
LocalDistinctEventTypes_std1
LocalDistinctEventTypes_std2
LocalDistinctEventTypes_std3
LocalDistinctEventTypes_std6
LocalDistinctEventTypes_std12
LocalDistinctEventTypes_std18
LocalDistinctEventSubtypes_std1
LocalDistinctEventSubtypes_std2
LocalDistinctEventSubtypes_std3
LocalDistinctEventSubtypes_std6
LocalDistinctEventSubtypes_std12
LocalDistinctEventSubtypes_std18
LocalTotalFatalities_std1
LocalTotalFatalities_std2
LocalTotalFatalities_std3
LocalTotalFatalities_std6
LocalTotalFatalities_std12
LocalTotalFatalities_std18
LocalMeanSeverity_std1
LocalMeanSeverity_std2
LocalMeanSeverity_std3
L

In [13]:
def make_year_blocked_folds(df, year_col="periodStart", n_folds=None):
    """
    Forward-chaining CV where each fold trains on one year and validates on the next.
    Assumes periodStart encodes 28-day periods that map to calendar years.
    """
    # Extract year from periodStart
    df["year"] = pd.to_datetime(df[year_col]).dt.year

    years = sorted(df["year"].unique())

    folds = []

    for i in range(len(years) - 1):
        train_year = years[i]
        val_year   = years[i + 1]

        train_idx = df[df["year"] == train_year].index
        val_idx   = df[df["year"] == val_year].index

        folds.append((train_idx, val_idx))

    return folds


In [14]:
folds = make_year_blocked_folds(df_train, year_col="periodStart")
#folds

In [15]:
gc.collect()

24

## Known good code for training LightGVM models. 

In [ ]:
import lightgbm as lgb
import joblib
import numpy as np
import gc
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import os

#folds = make_time_series_folds(df_train, "periodStart", n_folds=2)

models = {}
cv_results = {}

feature_cols = [
    c for c in df_train.columns
    if c not in target_columns
    and c not in ["id", "name", 'year', "periodStart", "periodEnd", "period_midpoint"]
]

for target_column in target_columns:

    print(f"\n=== Training {target_column} with Time-Series CV ===")

        #Skip iteration if the model already exists
    if os.path.isfile(f"../Models/CatBoost/catboost_model_{target_column}.pkl"):
        print("Model already exists, skipping...")
        break
    
    fold_metrics = []

    for fold_num, (train_idx, val_idx) in enumerate(folds, start=1):

        gc.collect() #garbage collection, no silver bullet but might help RAM issues.
        
        print(f"\nFold {fold_num}")

        # Full fold (past only)
        #df_train_fold = df_train.loc[train_idx]

        # Time-aware subsample to ~200k rows
        #df_train_fold_small = subsample_time_period_balanced(
        #    df_train_fold,
        #    target_column,
        #    max_rows=100_000
        #)

        #y_train = df_train_fold_small[target_column]
        #X_train = df_train_fold_small[feature_cols]

        # Validation stays full
        #df_val_fold = df_train.loc[val_idx]
        #y_val = df_val_fold[target_column]
        #X_val = df_val_fold[feature_cols]

        X_train = df_train.loc[train_idx, feature_cols]
        y_train = df_train.loc[train_idx, target_column]

        X_val = df_train.loc[val_idx, feature_cols]
        y_val = df_train.loc[val_idx, target_column]

        
        # Compute class imbalance weight
        #positives = int(y_train.sum())
        #negatives = len(y_train) - positives
        #scale_pos_weight = max(negatives / positives if positives > 0 else 1.0, 1.0)

        classes = np.unique(y_train)
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
        class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
        print("Class weights:", class_weight_dict)
        
        #print(f"scale_pos_weight={scale_pos_weight}, "f"train_rows={len(y_train)}, val_rows={len(y_val)}")

        # Build model
        model = lgb.LGBMClassifier(
            boosting_type='gbdt',
            objective="binary",
            learning_rate=0.01,
            n_estimators=100,   # smaller for CV
            #scale_pos_weight=scale_pos_weight,
            class_weight=class_weight_dict,
            random_state=1,
            n_jobs=5,
        )

        # Train on this fold
        model.fit(X_train, y_train)

        # Predict on validation fold
        y_pred = model.predict_proba(X_val)[:, 1]

        # Compute metric
        auc = roc_auc_score(y_val, y_pred)
        fold_metrics.append(auc)

        print(f"Fold {fold_num} AUC: {auc:.4f}")

        #feature importances: useful for identifying leakage.
        importances = model.feature_importances_
        feat_imp = pd.DataFrame({
            "feature": feature_cols,
            "importance": importances
        })
        
        feat_imp = feat_imp.sort_values("importance", ascending=False)
        top20 = feat_imp.head(20)
        print(top20)

        
        # Free memory
        #del df_train_fold, df_train_fold_small, df_val_fold
        del X_train, X_val, y_train, y_val, y_pred
        gc.collect()

    #end of fold loop
    
    # Store CV results
    cv_results[target_column] = {
        "mean_auc": np.mean(fold_metrics),
        "folds": fold_metrics
    }

    print(f"\n{target_column} CV mean AUC: {np.mean(fold_metrics):.4f}")

    # === Train final model on ALL df_train ===

    # Build final training set: all periods, subsampled by per-period quota
    #df_train_final = subsample_time_period_balanced(
    #    df_train,
    #    target_column,
    #    max_rows=200_000
    #)

    y_train_final = df_train[target_column]
    X_train_final = df_train[feature_cols]
    
    y_train_full = df_train[target_column]
   # positives = int(y_train_full.sum())
   # negatives = len(y_train_full) - positives
    # = max(negatives / positives if positives > 0 else 1.0, 1.0)

    X_train_full = df_train[feature_cols]

    classes = np.unique(y_train_full)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_full)
    class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
    print("Class weights:", class_weight_dict)
    
    final_model = lgb.LGBMClassifier(
        boosting_type='gbdt',
        objective="binary",
        learning_rate=0.01,
        n_estimators=500,
        #scale_pos_weight=scale_pos_weight,
        class_weight=class_weight_dict,
        random_state=1,
        n_jobs=5,
    )

    final_model.fit(X_train_full, y_train_full)

    joblib.dump(final_model, f"../Models/LightGBM/lightgbm_model_{target_column}.pkl")
    models[target_column] = final_model

    del X_train_full, y_train_full#, df_train_final

In [ ]:
gc.collect()

## Random Forest Models

In [59]:
from sklearn.ensemble import RandomForestClassifier
import joblib
import numpy as np
import gc
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os

models_rf = {}
cv_results = {}

feature_cols = [
    c for c in df_train.columns
    if c not in target_columns
    and c not in ["id", "name", 'year', "periodStart", 'latitude', 'longitude', "periodEnd", "period_midpoint"]
]

for target_column in target_columns:

    print(f"\n=== Training {target_column} for Random Forest with Time-Series CV ===")

    #Skip iteration if the model already exists
    if os.path.isfile(f"../Models/RF/rf_model_{target_column}.pkl"):
        print("Model already exists, skipping...")
        break
    
    fold_metrics = []

    for fold_num, (train_idx, val_idx) in enumerate(folds, start=1):

        gc.collect() #garbage collection, no silver bullet but might help RAM issues.
        
        print(f"\nFold {fold_num}")


        X_train = df_train.loc[train_idx, feature_cols]
        X_train['country'] = X_train['country'].astype('category').cat.codes #need to shift categorical col to the categorical codes; these will match the exported ones. 
        y_train = df_train.loc[train_idx, target_column]

        X_val = df_train.loc[val_idx, feature_cols]
        X_val['country'] = X_val['country'].astype('category').cat.codes
        y_val = df_train.loc[val_idx, target_column]
        
        classes = np.unique(y_train)
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
        class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
        print("Class weights:", class_weight_dict)

        #build model
        model_rf = RandomForestClassifier(
            n_estimators=100,      # number of trees
            max_depth=None,       # let trees grow fully
            random_state=42,      # reproducibility
            n_jobs=5,            # use all CPU cores
            class_weight=class_weight_dict, 
        )

        # Train on this fold
        model_rf.fit(X_train, y_train)

        # Predict on validation fold
        y_pred = model_rf.predict_proba(X_val)[:, 1]

        # Compute metric
        auc = roc_auc_score(y_val, y_pred)
        fold_metrics.append(auc)

        print(f"Fold {fold_num} AUC: {auc:.4f}")

        #feature importances: useful for identifying leakage.
        importances = model_rf.feature_importances_
        feat_imp = pd.DataFrame({
            "feature": feature_cols,
            "importance": importances
        })
        
        feat_imp = feat_imp.sort_values("importance", ascending=False)
        top20 = feat_imp.head(20)
        print(top20)

        
        # Free memory
        #del df_train_fold, df_train_fold_small, df_val_fold
        del X_train, X_val, y_train, y_val, y_pred
        gc.collect()

    #end of fold loop
    
    # Store CV results
    cv_results[target_column] = {
        "mean_auc": np.mean(fold_metrics),
        "folds": fold_metrics
    }

    print(f"\n{target_column} CV mean AUC: {np.mean(fold_metrics):.4f}")
    
    y_train_full = df_train[target_column]
    X_train_full = df_train[feature_cols]
    X_train_full['country'] = X_train_full['country'].astype('category').cat.codes

    classes = np.unique(y_train_full)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_full)
    class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
    print("Class weights:", class_weight_dict)
    
    final_model_rf = RandomForestClassifier(
        n_estimators=200,      # number of trees
        max_depth=None,       # let trees grow fully
        random_state=42,      # reproducibility
        n_jobs=5,            # use all CPU cores
        class_weight=class_weight_dict, 
    )

    final_model_rf.fit(X_train_full, y_train_full)

    joblib.dump(final_model, f"../Models/RF/rf_model_{target_column}.pkl")
    models_rf[target_column] = final_model_rf

    del X_train_full, y_train_full


=== Training TargetConflictLocal_t+1 for Random Forest with Time-Series CV ===

Fold 1
Class weights: {np.False_: np.float64(0.5297574772940808), np.True_: np.float64(8.901249794441704)}
Fold 1 AUC: 0.9999
                              feature  importance
368             LocalEventCount_mean1    0.075199
404            LocalMaxSeverity_mean1    0.070776
398           LocalMeanSeverity_mean1    0.054496
380     LocalDistinctEventTypes_mean1    0.046413
374     LocalDistinctActorCount_mean1    0.046109
386  LocalDistinctEventSubtypes_mean1    0.045654
399           LocalMeanSeverity_mean2    0.044300
400           LocalMeanSeverity_mean3    0.037994
381     LocalDistinctEventTypes_mean2    0.036170
388  LocalDistinctEventSubtypes_mean3    0.030250
387  LocalDistinctEventSubtypes_mean2    0.029200
375     LocalDistinctActorCount_mean2    0.027062
382     LocalDistinctEventTypes_mean3    0.025566
369             LocalEventCount_mean2    0.025260
406            LocalMaxSeverity_mean3    0.

C:\Users\andre\AppData\Local\Temp\ipykernel_3308\708603610.py:95: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train_full['country'] = X_train_full['country'].astype('category').cat.codes


Class weights: {np.False_: np.float64(0.5321781775799697), np.True_: np.float64(8.269240485378537)}

=== Training TargetConflictRegional_t+1 for Random Forest with Time-Series CV ===

Fold 1
Class weights: {np.False_: np.float64(0.9756574559743326), np.True_: np.float64(1.025588313312365)}
Fold 1 AUC: 0.9995
                                     feature  importance
482            RegionalTotalFatalities_mean1    0.117226
483            RegionalTotalFatalities_mean2    0.089808
488               RegionalMeanSeverity_mean1    0.059723
494                RegionalMaxSeverity_mean1    0.057047
123             RegionalTotalFatalities_std2    0.043181
489               RegionalMeanSeverity_mean2    0.028188
495                RegionalMaxSeverity_mean2    0.023163
124             RegionalTotalFatalities_std3    0.023000
490               RegionalMeanSeverity_mean3    0.020797
484            RegionalTotalFatalities_mean3    0.019800
487           RegionalTotalFatalities_mean18    0.016903
485   

C:\Users\andre\AppData\Local\Temp\ipykernel_3308\708603610.py:95: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train_full['country'] = X_train_full['country'].astype('category').cat.codes


Class weights: {np.False_: np.float64(1.2643414538640114), np.True_: np.float64(0.8270789497758668)}

=== Training TargetConflictLocal_t+3 for Random Forest with Time-Series CV ===

Fold 1
Class weights: {np.False_: np.float64(0.5595891614717406), np.True_: np.float64(4.695393823733518)}
Fold 1 AUC: 0.9214
                               feature  importance
400            LocalMeanSeverity_mean3    0.029321
407             LocalMaxSeverity_mean6    0.026426
399            LocalMeanSeverity_mean2    0.024429
17        LocalDistinctActorCount_std6    0.023381
403           LocalMeanSeverity_mean18    0.023193
408            LocalMaxSeverity_mean12    0.022039
41              LocalMeanSeverity_std6    0.019683
383      LocalDistinctEventTypes_mean6    0.018181
401            LocalMeanSeverity_mean6    0.017758
409            LocalMaxSeverity_mean18    0.016552
404             LocalMaxSeverity_mean1    0.016153
42             LocalMeanSeverity_std12    0.015083
402           LocalMeanSeveri

...

Class weights: {np.False_: np.float64(4.267787618576631), np.True_: np.float64(0.5663519352225175)}

=== Training TargetConflictLocal_t+12 for Random Forest with Time-Series CV ===

Fold 1
Class weights: {np.False_: np.float64(0.6357811527303052), np.True_: np.float64(2.341198096885813)}
Fold 1 AUC: 0.8788
                               feature  importance
1                             latitude    0.037202
2                            longitude    0.035276
4                 minCapitalDistanceKm    0.032397
3                  minBorderDistanceKm    0.031900
390  LocalDistinctEventSubtypes_mean12    0.017511
402           LocalMeanSeverity_mean12    0.017309
403           LocalMeanSeverity_mean18    0.016964
19       LocalDistinctActorCount_std18    0.015645
408            LocalMaxSeverity_mean12    0.015242
379     LocalDistinctActorCount_mean18    0.013602
385     LocalDistinctEventTypes_mean18    0.012431
31    LocalDistinctEventSubtypes_std18    0.012037
492        RegionalMeanSeveri

C:\Users\andre\AppData\Local\Temp\ipykernel_3308\708603610.py:95: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train_full['country'] = X_train_full['country'].astype('category').cat.codes


Class weights: {np.False_: np.float64(0.6746213585609351), np.True_: np.float64(1.931669081378502)}

=== Training TargetConflictRegional_t+12 for Random Forest with Time-Series CV ===

Fold 1
Class weights: {np.False_: np.float64(5.346024691358025), np.True_: np.float64(0.5515886764765826)}
Fold 1 AUC: 0.9627
                            feature  importance
1                          latitude    0.060254
487  RegionalTotalFatalities_mean18    0.051994
2                         longitude    0.049169
486  RegionalTotalFatalities_mean12    0.036329
483   RegionalTotalFatalities_mean2    0.031174
485   RegionalTotalFatalities_mean6    0.028101
4              minCapitalDistanceKm    0.027757
123    RegionalTotalFatalities_std2    0.027077
3               minBorderDistanceKm    0.025211
127   RegionalTotalFatalities_std18    0.021555
499      RegionalMaxSeverity_mean18    0.019106
0                           country    0.018080
482   RegionalTotalFatalities_mean1    0.017499
126   RegionalTot

C:\Users\andre\AppData\Local\Temp\ipykernel_3308\708603610.py:95: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train_full['country'] = X_train_full['country'].astype('category').cat.codes


Class weights: {np.False_: np.float64(8.192805606379673), np.True_: np.float64(0.5324978964492063)}


In [52]:
gc.collect()

53

## CatBoost Modelling

In [ ]:
from catboost import CatBoostClassifier
import joblib
import numpy as np
import gc
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os

models_catboost = {}
cv_results = {}

feature_cols = [
    c for c in df_train.columns
    if c not in target_columns
    and c not in ["id", "name", 'year', 'latitude', 'longitude', "periodStart", "periodEnd", "period_midpoint"]
]

for target_column in target_columns:

    print(f"\n=== Training {target_column} for Random Forest with Time-Series CV ===")

    #Skip iteration if the model already exists
    if os.path.isfile(f"../Models/CatBoost/catboost_model_{target_column}.pkl"):
        print("Model already exists, skipping...")
        break
    
    
    fold_metrics = []

    for fold_num, (train_idx, val_idx) in enumerate(folds, start=1):

        gc.collect() #garbage collection, no silver bullet but might help RAM issues.
        
        print(f"\nFold {fold_num}")


        X_train = df_train.loc[train_idx, feature_cols]
       # X_train['country'] = X_train['country'].astype('category').cat.codes #need to shift categorical col to the categorical codes; these will match the exported ones. 
        y_train = df_train.loc[train_idx, target_column]

        X_val = df_train.loc[val_idx, feature_cols]
        #X_val['country'] = X_val['country'].astype('category').cat.codes
        y_val = df_train.loc[val_idx, target_column]

        cat_features = [i for i, col in enumerate(X_train.columns)
                if X_train[col].dtype.name in ["category", "object"]]
        
        classes = np.unique(y_train)
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
        class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
        class_weights = [class_weight_dict[cls] for cls in classes]
        print("Class weights:", class_weight_dict)

        #build model
        model_catboost = CatBoostClassifier(
            iterations=500,
            learning_rate=0.05,
            depth=8,
            loss_function="Logloss",
            eval_metric="AUC",
            cat_features=cat_features,
            class_weights = class_weights,
            verbose=False
        )
        # Train on this fold
        model_catboost.fit(X_train, y_train)

        # Predict on validation fold
        y_pred = model_catboost.predict_proba(X_val)[:, 1]

        # Compute metric
        auc = roc_auc_score(y_val, y_pred)
        fold_metrics.append(auc)

        print(f"Fold {fold_num} AUC: {auc:.4f}")

        #feature importances: useful for identifying leakage.
        importances = model_catboost.feature_importances_
        feat_imp = pd.DataFrame({
            "feature": feature_cols,
            "importance": importances
        })
        
        feat_imp = feat_imp.sort_values("importance", ascending=False)
        top20 = feat_imp.head(20)
        print(top20)

        
        # Free memory
        #del df_train_fold, df_train_fold_small, df_val_fold
        del X_train, X_val, y_train, y_val, y_pred
        gc.collect()

    #end of fold loop
    
    # Store CV results
    cv_results[target_column] = {
        "mean_auc": np.mean(fold_metrics),
        "folds": fold_metrics
    }

    print(f"\n{target_column} CV mean AUC: {np.mean(fold_metrics):.4f}")
    
    y_train_full = df_train[target_column]
    X_train_full = df_train[feature_cols]
   # X_train_full['country'] = X_train_full['country'].astype('category').cat.codes
    cat_features = [i for i, col in enumerate(X_train.columns)
                if X_train[col].dtype.name in ["category", "object"]]
    
    classes = np.unique(y_train_full)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_full)
    class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
    class_weights = [class_weight_dict[cls] for cls in classes]
    print("Class weights:", class_weight_dict)


    
    final_model_catboost = CatBoostClassifier(
            iterations=500,
            learning_rate=0.05,
            depth=8,
            loss_function="Logloss",
            eval_metric="AUC",
            cat_features=cat_features,
            class_weights = class_weights
            verbose=False,
        )

    final_model_catboost.fit(X_train_full, y_train_full)

    joblib.dump(final_model, f"../Models/CatBoost/catboost_model_{target_column}.pkl")
    models_catboost[target_column] = final_model_catboost

    del X_train_full, y_train_full

In [ ]:
gc.collect()

## Load Models From File and Evaluate

In [53]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Model Evaluation:")

for target_column in target_columns:

    print(f"\nEvaluating model for: {target_column}")

    #build testing dataset (y; actual target values)
    #test_ids = subsamples_test[target_column]
    #df_test = #df[df["id"].isin(test_ids)].copy()

    y_test = df_test[[target_column]].copy()

    #build the testing dataset (x):
    feature_cols = [
        c for c in df_test.columns
        if c not in target_columns
        and c not in ["id", "placeName", "name", "year", 'latitude', 'longitude', "periodStart", "periodEnd", "period_midpoint"]
    ]

    X_test = df_test[feature_cols].copy()
    
    print("X_test shape:", X_test.shape)
    print("y_test positives:", y_test[target_column].sum())

    #obtain model from dict
    #model = models[target_column]

    #load model from disk (pickle files are neat, this saves tedious mucking around with only saving part of the model):
    model = joblib.load(f"../Models/LightGBM/lightgbm_model_{target_column}.pkl")
    
    #predict
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_prob >= 0.5).astype(int)

    #Conf matrix
    cm = confusion_matrix(y_test, y_pred) #note this all breaks for regional_12, as there are 0 positive instances. 
    tn, fp, fn, tp = cm.ravel()

    #specificity = TN / (TN + FP)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    #Classification metrics:
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_pred_prob) if y_test[target_column].nunique() > 1 else float("nan")

    print("\nMetrics:")
    print(f"Accuracy:      {accuracy:.4f}")
    print(f"Precision:     {precision:.4f}")
    print(f"Recall:        {recall:.4f}")
    print(f"Specificity:   {specificity:.4f}")
    print(f"F1 Score:      {f1:.4f}")
    print(f"AUC:           {auc:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    print("\nConfusion Matrix:")
    print(cm)

    print("\nFeature Importance (by gain):")
    importance = model.feature_importances_
    #importance = model.feature_importance(importance_type='gain')

    
    fi_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": importance
    }).sort_values("importance", ascending=False)

    print(fi_df.head(20))  # top 20 features
    
    #Delete iteration variables to free up memory:
    del X_test, y_test, y_pred, y_pred_prob, cm, fi_df #df_test, X_test, y_test, y_pred, y_pred_prob, cm, fi_df
    gc.collect()

    print(f"End of evaluation for model {target_column}")
    print("------")
    #end of this iteration
    

Model Evaluation:

Evaluating model for: TargetConflictLocal_t+1
X_test shape: (199848, 564)
y_test positives: 18815

Metrics:
Accuracy:      1.0000
Precision:     1.0000
Recall:        1.0000
Specificity:   1.0000
F1 Score:      1.0000
AUC:           1.0000

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    181033
         1.0       1.00      1.00      1.00     18815

    accuracy                           1.00    199848
   macro avg       1.00      1.00      1.00    199848
weighted avg       1.00      1.00      1.00    199848


Confusion Matrix:
[[181033      0]
 [     0  18815]]

Feature Importance (by gain):
                                               feature  importance
392                         LocalTotalFatalities_mean1        2960
398                            LocalMeanSeverity_mean1        2000
404                             LocalMaxSeverity_mean1        1000
562  Internally_displaced_persons_new